# Mission 3 — 환자 증상 인식 (최종 제출)

119 신고 대화의 **전사 텍스트만** 읽고 9개 증상을 예측합니다. 평가 지표는
증상별 F1의 평균(Macro F1)이고, 판정 임계값은 **전 증상 0.5**입니다.

공식 Validation 3,640통화 실측 **Macro F1 = 0.65746**

이 노트북은 제출 모델의 **실제 학습 로그**를 담고 있습니다. 5절의 출력은
학습이 GPU에서 남긴 기록을 멤버별로 그대로 옮긴 것입니다.

## 1. 최종 구성

신경망 16개가 각자 9개 증상의 확률을 내고, 그 확률을 단순 평균한 뒤 TF-IDF
선형모델의 확률을 10% 섞어 0.5에서 판정합니다.

```
최종 확률 = 0.9 × (신경망 16개 확률의 평균) + 0.1 × (TF-IDF 확률)
판정      = 최종 확률 >= 0.5
```

### 멤버 — 11개 계열 16개

| 계열 | 멤버 | 무엇이 다른가 |
|---|---:|---|
| base mean lr2e-5 | 2 | 기준 레시피 |
| base attention lr2e-5 | 2 | 증상별 attention 헤드 |
| base mean lr3e-5 | 1 | 학습률 |
| base attention lr3e-5 | 1 | 학습률 + 헤드 |
| base 정규화 텍스트 | 2 | 동의어 통일 입력 |
| base 도메인 적응 | 2 | 전사로 MLM 추가 학습 후 미세조정 |
| base CLS 풀링 | 1 | `[CLS]` 토큰 사용 |
| large mean | 2 | `klue/roberta-large` |
| large attention | 1 | large + attention 헤드 |
| large 정규화 텍스트 | 1 | large + 동의어 통일 입력 |
| large TAPT | 1 | 도메인 적응 사전학습된 large |

쓴 사전학습 체크포인트는 세 개입니다(모두 오픈소스).

| 표기 | 사전학습 모델 | 파라미터 |
|---|---|---:|
| base | `klue/roberta-base` | 110M |
| large | `klue/roberta-large` | 337M |
| large TAPT | `SangwonYoon/klue-roberta-large-tapt` | 337M |

`base 도메인 적응` 멤버는 `klue/roberta-base`를 Training 전사 텍스트로
MLM 추가 학습한 뒤 미세조정한 것입니다(라벨 미사용).

공통: 3 epoch · AdamW · bf16 · `pos_weight = clip(sqrt(음성/양성), 1, 5)` ·
**Training 29,200통화 전부** · 멤버 가중치 없음 · 후처리 없음.

`klue/roberta-large` mean·attention 멤버는 3 epoch 스케줄에서 **epoch 2
가중치**를 씁니다. 홀드아웃에서 epoch 2가 최고였고 이후 하락하기 때문입니다.
스케줄을 2 epoch로 줄이면 학습률이 더 짧은 구간에 걸쳐 감쇠해 다른 모델이
되므로(홀드아웃 −0.00198), 스케줄은 3 epoch로 두고 저장 시점만 당겼습니다.

체크포인트마다 자기 전처리(`normalize`)와 인코더 설정을 기록하고 있어서,
추론 코드가 멤버별로 맞는 입력을 만들어 넣고 **네트워크 없이** 동작합니다.

## 2. 어떻게 이 구성에 도달했는가

약 110회의 짝비교를 **Training 내부 홀드아웃 5,840통화**에서 수행했습니다.
공식 Validation은 설계 과정에서 쓰지 않고, 제출 결정 시점에만 측정했습니다.

### 효과가 있었던 것

| 방법 | Δ Macro F1 | 검증 방식 |
|---|---:|---|
| **앙상블 (확률 단순 평균)** | **+0.0089** | 크기별 부분집합 150개 평균 |
| **large 계열 5개 추가** | +0.0037 | 반분 검증 240회 전승 |
| **이질 구성 (계열 섞기)** | +0.0016 | 무작위 조합 200개 중 200개 승 |
| **TF-IDF 10% 혼합** | +0.0012 | 반분 검증 80회, 91% 승 |
| **전체 데이터 학습** | +0.0010 | 데이터 크기 곡선 |

앙상블은 **10개에서 포화**합니다(10개 0.6462 / 16개 0.6463 / 23개 0.6462).
그래서 멤버를 늘리는 대신 **계열을 섞었습니다**. 같은 6개라도 한 레시피로
채우면 0.6448, 계열마다 하나씩 뽑으면 0.6464입니다. 멤버 단독 품질은
같고(0.6374 vs 0.6381) 상호 불일치만 높습니다(0.0295 → 0.0358).

`large` 계열은 처음에 "base와 비슷한데 3.7배 느리다"고 보고 두 번 제외했는데,
그 판단이 틀렸습니다. 설정이 잘못된 상태의 측정이었고(올바른 값은 lr 1.5e-5,
batch 4×8), 더 중요하게는 **단독 성능으로 판단한 것 자체가 잘못**이었습니다 —
단독 성능과 앙상블 기여의 상관은 +0.22뿐입니다. 판단 기준을 "앙상블에 넣었을
때의 변화"로 바꾸고 백본 후보 15개를 다시 재서 large 계열만 통과했습니다.

멤버 수가 늘어 오른 것이 아니라는 통제: large 5개 추가는 +0.0019(승률 98%),
기존 멤버 3개를 복제해 추가하면 **−0.0012(승률 0%)**입니다.

### 효과가 없었던 것

| 축 | 시도 | 결과 |
|---|---|---|
| 백본 | 10종 (KcELECTRA, km-bert(의학), kobigbird 등) | 전부 base 이하 |
| 백본 | large 추가 레시피 6종 | 전부 기준 미달 |
| 구조 | 풀링 층 변경 · 하위 층 동결 · meanmax 헤드 | 전부 음수 |
| 손실 | ASL · Focal · soft macro-F1 · R-Drop | 전부 음수 |
| 라벨 | 자기 증류 · 라벨 스무딩 · 증상별 손실 가중치 | 전부 음수 |
| 분포 | 희귀 증상 오버샘플링 | −0.0109 |
| 결합 | 스태킹 · 멤버 가중치 · 증상별 혼합비율 | 전부 음수 |
| 압축 | 가중치 평균(model soup) | −0.0091 |

**공통 교훈: 조정할 수 있는 값을 늘리면 거의 항상 손해였습니다.** 홀드아웃
5,840통화로 그 값을 맞추면 그 데이터에만 맞춰집니다. 실제로 측정했습니다 —
홀드아웃에서 "최적의 5개 조합"은 0.65231이지만, 고르는 것까지 반쪽 데이터
안에서 하고 다른 쪽에서 재면 0.64921입니다. **차이 +0.0031이 순수한 선택
편향**입니다.

## 3. 데이터 분석이 알려준 것

### 모델은 전사에서 읽을 수 있는 것을 거의 다 읽습니다

놓친 정답(FN)과 잘못 짚은 것(FP)에 전사 근거가 있는지 셌습니다.

| 증상 | FN 수 | FN에 표현 있음 | FP 수 | FP에 표현 있음 |
|---|---:|---:|---:|---:|
| 구토 | 404 | **1.5%** | 280 | 50.4% |
| 두통 | 309 | 10.4% | 240 | 56.2% |
| 호흡곤란 | 210 | 33.3% | 313 | **86.9%** |

**양쪽 다 어노테이션 행동입니다.** 구토는 놓친 404건 중 6건에만 구토 표현이
있습니다 — 읽고도 못 맞힌 게 아니라 읽을 것이 없습니다. 반대로 호흡곤란 FP의
87%에 "숨"이 나옵니다.

### 오심만 질적으로 다릅니다

표현이 있는 통화 안에서 라벨 유무를 구분할 수 있는가(조건부 AUC):

| 열상 | 고열 | 복통 | 호흡곤란 | 전신쇠약 | 두통 | 어지러움 | 구토 | **오심** |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 0.980 | 0.940 | 0.897 | 0.871 | 0.842 | 0.777 | 0.774 | 0.726 | **0.567** |

"메스꺼워요"가 전사에 또박또박 있어도 오심 라벨은 **63%만** 붙습니다.
최종 성적에서도 오심 F1이 0.4101로 최하위이고, 이는 모델의 약점이 아니라
라벨의 성질을 반영합니다.

### 512 토큰 한도는 충분합니다

토큰 길이 중앙값 230, 95분위 460. 한도를 넘는 통화는 788개(2.7%)이고 전체
토큰 중 버려지는 비율은 0.8%입니다. 버려지는 구간에만 증상 근거가 있는 양성은
1,115건 중 **10건**입니다. 이 방향에는 얻을 것이 없습니다.

### 임계값 0.5 고정의 비용은 +0.0062

증상별 임계값을 자유롭게 골랐다면 0.6441 → 0.6503입니다. 단조 보정은 순위를
못 바꾸므로 **확률 보정으로 얻을 수 있는 것도 이게 전부**입니다.

## 4. 모델 코드

In [1]:
"""Fine-tune a Korean encoder for the nine symptoms, on this machine's GPU.

Deliberately independent of ver4: weights start from the public pretrained
checkpoint, the split is our own Training-internal grouped split, and official
Validation is never read. That keeps every result here free of the anchoring
that ver5-ver8 ran into.

Decisions are scored at the mandated 0.5 threshold for every symptom.
"""
from __future__ import annotations

import argparse
import gc
import json
import sys
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import (AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup,
                          get_linear_schedule_with_warmup)

sys.path.insert(0, str(Path(__file__).parent))
from common_local import TARGETS, TRAIN_DIR, load, macro_f1, split          # noqa: E402
from normalize import VARIANTS, normalize                                    # noqa: E402

OUT = Path(__file__).resolve().parents[3] / "results" / "mission3" / "local"


class Calls(Dataset):
    """통화 하나 = 전사 텍스트 1개 + 증상 9개 타깃.

    타깃은 0/1 하드 라벨일 수도, 교사 확률을 섞은 소프트 타깃일 수도 있습니다
    (`--soft_targets`). `aux` 는 보조 과제용이며 기본은 쓰지 않습니다.
    """

    def __init__(self, texts, Y=None, aux=None):
        self.texts, self.Y, self.aux = texts, Y, aux

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        y = self.Y[i] if self.Y is not None else 0
        return self.texts[i], y, (self.aux[i] if self.aux is not None else -100)


class Collate:
    """배치 안에서 가장 긴 전사에 맞춰 토큰화합니다(동적 패딩).

    512 로 고정 패딩하면 중앙값이 230 토큰인 이 데이터에서 절반 이상이
    낭비됩니다. 배치마다 길이를 맞추면 같은 시간에 더 많이 학습합니다.
    """

    def __init__(self, tokenizer, max_length):
        self.tok, self.max_length = tokenizer, max_length

    def __call__(self, batch):
        enc = self.tok([b[0] for b in batch], padding=True, truncation=True,
                       max_length=self.max_length, return_tensors="pt")
        enc["labels"] = torch.tensor(np.stack([b[1] for b in batch]), dtype=torch.float32)
        enc["aux"] = torch.tensor([b[2] for b in batch], dtype=torch.long)
        return enc


class Classifier(nn.Module):
    """Mean pooling, or one attention query per symptom (LAAT / PLM-ICD style).

    Mean pooling forces all nine symptoms to share a single sentence vector.
    Label attention lets each symptom weight the tokens itself, so "열상" can
    look at "피가 나요" while "오심" looks somewhere else entirely -- which is
    the pooling ver4 used and this pipeline did not.
    """

    def __init__(self, name, n_labels, dropout=0.1, head="mean", attn_dim=256,
                 label_corr=False, aux_dim=0, pool_layer=-1, freeze_layers=0):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(name)
        self.drop = nn.Dropout(dropout)
        self.head_kind = head
        # Which transformer layer to read. The last one is tuned hardest to the
        # training labels; an earlier layer carries more general features and
        # is wrong about different calls, which is what the ensemble wants.
        self.pool_layer = pool_layer
        # Freezing the bottom layers leaves the general features alone and
        # adapts only the upper ones -- a different model from the same weights.
        if freeze_layers:
            for p_ in self.encoder.embeddings.parameters():
                p_.requires_grad = False
            for layer in self.encoder.encoder.layer[:freeze_layers]:
                for p_ in layer.parameters():
                    p_.requires_grad = False
        hidden = self.encoder.config.hidden_size
        # Nine independent sigmoids ignore that the labels travel together:
        # 오심 shows up beside 구토 and 어지러움 far more often than chance.
        # A zero-initialised residual mixer starts as the identity, so the run
        # only diverges from the plain head if the correlation actually pays.
        self.corr = None
        if label_corr:
            self.corr = nn.Linear(n_labels, n_labels)
            nn.init.zeros_(self.corr.weight)
            nn.init.zeros_(self.corr.bias)
        # Auxiliary softmax over whole label combinations. Only 188 of the 511
        # possible sets ever occur, and 어지러움+오심 is the ninth most common --
        # structure nine independent sigmoids cannot represent. The head is
        # dropped at inference, so it only shapes the encoder.
        self.aux = nn.Linear(hidden, aux_dim) if aux_dim else None
        if head == "meanmax":
            self.head = nn.Linear(hidden * 2, n_labels)
        elif head in ("mean", "cls", "max"):
            self.head = nn.Linear(hidden, n_labels)
        else:
            self.project = nn.Linear(hidden, attn_dim)
            self.queries = nn.Linear(attn_dim, n_labels, bias=False)
            self.out_weight = nn.Parameter(torch.empty(n_labels, hidden))
            self.out_bias = nn.Parameter(torch.zeros(n_labels))
            nn.init.xavier_uniform_(self.out_weight)

    def forward(self, input_ids, attention_mask, **kw):
        """(배치, 9) 로짓을 돌려줍니다.

        `pool_layer` 가 -1 이면 마지막 층을, 아니면 그 층의 출력을 씁니다.
        헤드 종류에 따라 토큰을 모으는 방식이 달라집니다 -- 추론 코드의
        같은 이름 함수와 **정확히 일치해야** 합니다. 학습기에만 있는 헤드를
        만들면 추론에서 state_dict 적재가 실패합니다.
        """
        if self.pool_layer == -1:
            hidden = self.encoder(input_ids=input_ids,
                                  attention_mask=attention_mask).last_hidden_state
        else:
            out = self.encoder(input_ids=input_ids, attention_mask=attention_mask,
                               output_hidden_states=True)
            hidden = out.hidden_states[self.pool_layer]
        mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-6)
        if self.head_kind == "meanmax":
            mx = hidden.masked_fill(mask == 0, float("-inf")).max(1).values
            logits = self.head(self.drop(torch.cat([pooled, mx], dim=-1)))
            if self.corr is not None:
                logits = logits + self.corr(logits)
            return (logits, self.aux(self.drop(pooled))) if self.aux is not None else logits
        if self.head_kind in ("mean", "cls", "max"):
            # Three ways to collapse the tokens into one vector. They score
            # about the same on their own, but they are wrong about different
            # calls -- which is what an ensemble wants. "cls" reads the
            # position the encoder was pretrained to summarise with; "max"
            # takes the strongest activation per dimension, so a single
            # decisive phrase can carry the whole call.
            if self.head_kind == "cls":
                pooled = hidden[:, 0]
            elif self.head_kind == "max":
                pooled = hidden.masked_fill(mask == 0, float("-inf")).max(1).values
            logits = self.head(self.drop(pooled))
        else:
            # scores: (batch, t

... (중략) ...

           # inference can build the architecture without fetching
                    # 440MB of pretrained weights it immediately overwrites.
                    "encoder_config": model.encoder.config.to_dict(),
                    "trained_on": int(len(tr)), "epochs": args.epochs,
                    "lr": args.lr, "seed": args.seed,
                    "saved_at_epoch": snapshot_at or args.epochs},
                   ckpt_dir / f"{tag}.ckpt")
        tok.save_pretrained(ckpt_dir / "tokenizer")
        log(f"  가중치 저장: submission_models/{tag}.ckpt")
    (OUT / f"history_{tag}.json").write_text(
        json.dumps({"args": vars(args), "history": history, "best": best},
                   ensure_ascii=False, indent=2), encoding="utf-8")
    log(f"  최고 epoch {best['epoch']}: {best['best_macro_f1']:.5f} "
        f"(그대로 {best['macro_f1']:.5f} / 강제 {best['macro_f1_forced']:.5f} 중 높은 쪽)")
    return best


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--model", default="klue/roberta-base")
    p.add_argument("--normalize", default="raw", choices=list(VARIANTS))
    p.add_argument("--epochs", type=int, default=3)
    p.add_argument("--batch_size", type=int, default=16)
    p.add_argument("--eval_batch_size", type=int, default=64)
    p.add_argument("--accum", type=int, default=2)
    p.add_argument("--lr", type=float, default=3e-5)
    p.add_argument("--max_length", type=int, default=512)
    p.add_argument("--dropout", type=float, default=0.1)
    p.add_argument("--head", default="mean",
                   choices=("mean", "cls", "max", "meanmax", "label_attention"))
    p.add_argument("--pos_weight", default="sqrt5",
                   choices=("none", "pow025", "pow035", "sqrt3", "sqrt5", "sqrt8", "sqrt",
                            "pow065", "pow080", "full"))
    p.add_argument("--loss", default="bce", choices=("bce", "asl", "focal"))
    p.add_argument("--gamma_neg", type=float, default=2.0, help="ASL/focal focusing strength.")
    p.add_argument("--ema", type=float, default=0.0, help="EMA decay; 0 disables.")
    p.add_argument("--train_frac", type=float, default=1.0, help="Fraction of the train split to use.")
    p.add_argument("--bag_seed", type=int, default=0, help="Which subsample, when train_frac < 1.")
    p.add_argument("--soft_targets", default="",
                   help="(29200, 9) teacher probabilities aligned with load() order.")
    p.add_argument("--soft_alpha", type=float, default=0.3,
                   help="Mixing weight: target = (1-a)*hard + a*soft.")
    p.add_argument("--label_weight", default="none",
                   help="none | invfreq | custom:w1,...,w9 -- per-symptom loss scale.")
    p.add_argument("--pool_layer", type=int, default=-1,
                   help="Transformer layer to pool from; -1 is the last.")
    p.add_argument("--freeze_layers", type=int, default=0,
                   help="Freeze embeddings and the bottom N encoder layers.")
    p.add_argument("--scheduler", default="linear", choices=("linear", "cosine"))
    p.add_argument("--warmup", type=float, default=0.1,
                   help="Fraction of steps spent warming up.")
    p.add_argument("--softf1", type=float, default=0.0,
                   help="Weight of a differentiable macro-F1 term added to BCE; 0 disables.")
    p.add_argument("--oversample", default="none", choices=("none", "log", "sqrt", "full"),
                   help="Draw calls carrying rare symptoms more often.")
    p.add_argument("--rdrop", type=float, default=0.0,
                   help="R-Drop strength; 0 disables. Costs a second forward pass.")
    p.add_argument("--aux", default="none",
                   help="none | count | combo:K -- training-only auxiliary head.")
    p.add_argument("--aux_weight", type=float, default=0.3)
    p.add_argument("--label_corr", action="store_true",
                   help="Zero-initialised residual mixer over the nine logits.")
    p.add_argument("--fold", type=int, default=-1,
                   help="With --folds k, train on all but this fold of the train split.")
    p.add_argument("--folds", type=int, default=0, help="Number of OOF folds; 0 disables.")
    p.add_argument("--llrd", type=float, default=1.0,
                   help="Layer-wise LR decay; 1.0 keeps every layer at --lr.")
    p.add_argument("--soft_debias", action="store_true",
                   help="With --soft_targets and a pos_weight, invert the weight's "
                        "effect on the target so the loss optimum equals the "
                        "teacher probability.")
    p.add_argument("--save_at_epoch", type=int, default=0,
                   help="Ship the weights from the end of this epoch rather than the last. "
                        "Fix it from a holdout run; never pick it from this run's output.")
    p.add_argument("--stop_at_save", action="store_true",
                   help="With --save_at_epoch, skip the remaining epochs.")
    p.add_argument("--full_train", action="store_true",
                   help="Train on all 29,200 calls; no holdout is left to score on.")
    p.add_argument("--arch_name", default="",
                   help="Public checkpoint name to record in the saved file. A domain-adapted run trains from a local directory, and that path will not exist on the grading machine -- the weights live in the state_dict, so only the architecture name needs to be portable.")
    p.add_argument("--save_model", action="store_true", help="Write a submission checkpoint.")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--grad_checkpoint", action="store_true")
    p.add_argument("--tag_suffix", default="", help="Keeps runs that differ only by epochs from overwriting each other.")
    p.add_argument("--log", default=str(OUT / "train_log.md"))
    args = p.parse_args()

    OUT.mkdir(parents=True, exist_ok=True)
    logfile = Path(args.log)

    def log(line):
        """화면과 로그 파일에 동시에 적습니다 (제출 노트북이 이 로그를 씁니다)."""
        print(line, flush=True)
        with logfile.open("a", encoding="utf-8") as handle:
            handle.write(line + "\n")

    log(f"\n### {time.strftime('%H:%M:%S')} {args.model} / {args.normalize} / seed {args.seed}")

    # 8GB is tight for the large encoder. Rather than lose a queued run to an
    # out-of-memory crash, halve the per-step batch and keep the effective batch
    # the same by doubling accumulation, then try again.
    while True:
        try:
            run(args, log)
            break
        except torch.cuda.OutOfMemoryError:
            gc.collect()          # drop the traceback's hold on the dead model
            torch.cuda.empty_cache()
            if args.batch_size <= 1:
                log("  VRAM 부족: batch 1에서도 실패해 이 설정은 건너뜁니다.")
                raise
            args.batch_size //= 2
            args.accum *= 2
            args.eval_batch_size = max(1, args.eval_batch_size // 2)
            log(f"  VRAM 부족 → batch {args.batch_size} / accum {args.accum} 로 재시도")

## 5. 실제 학습 로그

제출하는 16개 멤버가 학습 중 남긴 기록입니다. 멤버별로 해당 블록만 뽑았습니다.

아래 `Macro F1` 값은 **학습에 포함된 통화로 잰 값**입니다(전체 데이터로
학습하므로 남는 홀드아웃이 없습니다). 일반화 성능이 아니라 수렴 확인용이며,
실제 성능은 7절의 공식 Validation 실측값입니다. 구성은 모두 홀드아웃 실험에서
먼저 확정했습니다.

In [1]:
# 학습이 남긴 기록을 멤버별로 출력합니다.
for name in ('final_train_log.md', 'final_diverse_log.md', 'labelside_log.md'):
    print(open(f'results/mission3/local/{name}', encoding='utf-8').read())

----- base mean lr2e-5 · seed 42
### 20:55:08 klue/roberta-base / raw / seed 42
모델 klue/roberta-base | 전처리 raw | 손실 bce | 학습 29,200 / 자체검증 5,840
  epoch 1: loss 0.5036 | Macro F1 0.65159(학습에 포함됨·참고용) | 최소1개 강제 0.65084 | 빈 예측 189 | 5.7분
  epoch 2: loss 0.4145 | Macro F1 0.67587(학습에 포함됨·참고용) | 최소1개 강제 0.67540 | 빈 예측 103 | 5.7분
  epoch 3: loss 0.3827 | Macro F1 0.70337(학습에 포함됨·참고용) | 최소1개 강제 0.70245 | 빈 예측 86 | 5.7분
  가중치 저장: submission_models/roberta-base_raw_s42_FINAL.ckpt
  최고 epoch 3: 0.70337 (그대로 0.70337 / 강제 0.70245 중 높은 쪽)

----- base mean lr2e-5 · seed 1
### 21:18:32 klue/roberta-base / raw / seed 1
모델 klue/roberta-base | 전처리 raw | 손실 bce | 학습 29,200 / 자체검증 5,840
  epoch 1: loss 0.5033 | Macro F1 0.62910(학습에 포함됨·참고용) | 최소1개 강제 0.62701 | 빈 예측 162 | 5.7분
  epoch 2: loss 0.4147 | Macro F1 0.67753(학습에 포함됨·참고용) | 최소1개 강제 0.67703 | 빈 예측 124 | 5.7분
  epoch 3: loss 0.3848 | Macro F1 0.69742(학습에 포함됨·참고용) | 최소1개 강제 0.69760 | 빈 예측 75 | 5.7분
  가중치 저장: submission_models/roberta-base_raw_s1_FIN

## 6. 추론

제출 규격(출제문제 11쪽)에 맞춘 한 줄 실행입니다.

```bash
python inference.py --audio_dir {wav folder} --label_dir {json folder} \
    --ckpt_path ./weights --output ./outputs/mission3.csv
```

Mission 3은 음성을 읽지 않지만 `--audio_dir`를 받습니다(채점 인터페이스
호환). `--ckpt_path`에 폴더를 주면 안의 모든 체크포인트를 앙상블합니다.

`--device cpu`로 CPU 실행도 됩니다. **네트워크가 필요하지 않습니다** —
체크포인트가 인코더 설정을 직접 들고 있어서 사전학습 가중치를 받지 않습니다
(`HF_HUB_OFFLINE=1`에서 검증 완료).

In [1]:
"""Mission 3 submission inference.

Required interface (출제문제 11쪽):

    python inference.py --audio_dir {wav folder} --label_dir {json folder} \
        --ckpt_path {checkpoint file} --output ./outputs/mission3.csv

One command produces the CSV. Mission 3 reads no audio, but --audio_dir is
accepted because the organisers' harness passes it.

What this does:
  * reads only utterances[].text from each annotation JSON
  * averages the probabilities of every checkpoint in the ensemble
  * calls a symptom present when its probability is >= 0.5 -- the same
    threshold for all nine, as the organisers require
  * writes `label file name,symptom` with the symptom list as a string

Pass a directory to --ckpt_path to ensemble every .ckpt inside it, or a single
.ckpt file for one model. Both work; the submitted ensemble is a directory.

If a `tfidf.pkl` sits beside the checkpoints it is blended in at 10%, which
is worth +0.0011 Macro F1 and +0.0026 AP on the holdout. See tfidf_model.py
for the split-half validation behind that weight.
"""
from __future__ import annotations

import argparse
import csv
import json
import sys
from pathlib import Path

import numpy as np
import torch
from torch import nn
from transformers import AutoConfig, AutoModel, AutoTokenizer

TARGETS = ("고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란")
THRESHOLD = 0.5
UTTERANCE_KEYS = ("utterances", "utterences")


class Classifier(nn.Module):
    """Must match the architecture the checkpoints were trained with."""

    def __init__(self, name, n_labels, dropout=0.1, head="mean", attn_dim=256,
                 encoder_config=None):
        """체크포인트에 기록된 설정 그대로 모델을 세웁니다.

        Args:
            name: 사전학습 모델 이름. `encoder_config` 가 있으면 쓰이지
                않습니다(토크나이저 경로 대비용으로만 보관).
            n_labels: 증상 수 (9).
            dropout: 학습 때 쓴 값. 추론에서는 eval() 이라 동작하지 않지만,
                구조가 같아야 state_dict 가 들어맞습니다.
            head: 토큰들을 한 벡터로 모으는 방식. "mean" / "cls" / "max" /
                "label_attention" 중 하나이며, 멤버마다 다릅니다.
            attn_dim: label_attention 헤드의 중간 차원. 학습기와 추론기 모두
                256 고정이라 어긋날 수 없습니다.
            encoder_config: 체크포인트가 들고 있는 인코더 설정. 이것이 있으면
                사전학습 가중치를 받지 않고 구조만 만들어 **오프라인으로**
                동작합니다.
        """
        super().__init__()
        # from_pretrained downloads the published weights and load_state_dict
        # then replaces every one of them. Building from the stored config
        # skips that: no network, and eleven checkpoints load in seconds.
        if encoder_config:
            cfg = dict(encoder_config)
            self.encoder = AutoModel.from_config(
                AutoConfig.for_model(cfg.pop("model_type"), **cfg))
        else:
            self.encoder = AutoModel.from_pretrained(name)
        self.drop = nn.Dropout(dropout)
        self.head_kind = head
        hidden = self.encoder.config.hidden_size
        if head in ("mean", "cls", "max"):
            self.head = nn.Linear(hidden, n_labels)
        else:
            self.project = nn.Linear(hidden, attn_dim)
            self.queries = nn.Linear(attn_dim, n_labels, bias=False)
            self.out_weight = nn.Parameter(torch.empty(n_labels, hidden))
            self.out_bias = nn.Parameter(torch.zeros(n_labels))
            nn.init.xavier_uniform_(self.out_weight)

    def forward(self, input_ids, attention_mask, **kw):
        """(배치, 9) 로짓을 돌려줍니다. 확률로 바꾸는 것은 호출자 쪽입니다.

        헤드 종류에 따라 토큰을 모으는 방식이 다릅니다. mean 은 패딩을 제외한
        평균, cls 는 첫 토큰, max 는 차원별 최댓값이고, label_attention 은
        증상마다 다른 가중치로 토큰을 모읍니다 -- "숨"과 "피"에 주목해야 하는
        위치가 다르기 때문입니다.
        """
        hidden = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
        if self.head_kind in ("mean", "cls", "max"):
            # Three ways to collapse tokens into one vector. They must match
            # train_local.py exactly -- a head the trainer knows and this file
            # does not fails at load_state_dict, which is how "cls" was caught.
            if self.head_kind == "cls":
                pooled = hidden[:, 0]
            elif self.head_kind == "max":
                pooled = hidden.masked_fill(mask == 0, float("-inf")).max(1).values
            else:
                pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-6)
            return self.head(self.drop(pooled))
        scores = self.queries(torch.tanh(self.project(hidden)))
       

... (중략) ...

which is fine only
    # while every member uses the same vocabulary. A heterogeneous ensemble can
    # break that quietly -- wrong token ids produce plausible-looking garbage
    # rather than an error -- so the vocabulary size is checked against the
    # checkpoint's own embedding matrix before the shared copy is trusted.
    vocab_rows = blob["state_dict"]["encoder.embeddings.word_embeddings.weight"].shape[0]
    tokenizer_dir = ckpt_path.parent / "tokenizer"
    tok = None
    if tokenizer_dir.exists():
        shared = AutoTokenizer.from_pretrained(str(tokenizer_dir), use_fast=True)
        if len(shared) <= vocab_rows:
            tok = shared
        else:
            print(f"    공유 토크나이저 불일치 ({len(shared)} vs {vocab_rows}) — "
                  f"{blob['model_name']}에서 직접 불러옵니다", flush=True)
    if tok is None:
        tok = AutoTokenizer.from_pretrained(blob["model_name"], use_fast=True)
    model = Classifier(blob["model_name"], len(blob["target_symptoms"]),
                       blob.get("dropout", 0.1), blob.get("head", "mean"),
                       encoder_config=blob.get("encoder_config"))
    model.load_state_dict(blob["state_dict"])
    model.to(device).eval()
    amp = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else None

    out = []
    for start in range(0, len(texts), batch_size):
        batch = tok(texts[start:start + batch_size], padding=True, truncation=True,
                    max_length=blob.get("max_length", 512), return_tensors="pt")
        batch = {k: v.to(device) for k, v in batch.items()}
        if amp is not None:
            with torch.autocast("cuda", dtype=amp):
                logits = model(**batch)
        else:
            logits = model(**batch)
        out.append(torch.sigmoid(logits.float()).cpu().numpy())
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return np.concatenate(out)


def main():
    """출제문제 11쪽의 실행 규격 한 줄을 처리합니다.

    주석 JSON 을 읽고, 앙상블 멤버마다 확률을 계산해 평균하고, TF-IDF 를
    10% 섞은 뒤 전 증상 0.5 에서 판정해 CSV 로 저장합니다.
    """
    parser = argparse.ArgumentParser()
    parser.add_argument("--audio_dir", default="", help="미사용 (대회 인터페이스 호환).")
    parser.add_argument("--label_dir", required=True)
    parser.add_argument("--ckpt_path", required=True,
                        help=".ckpt 파일 또는 앙상블 체크포인트가 든 폴더.")
    parser.add_argument("--output", default="./outputs/mission3.csv")
    parser.add_argument("--tfidf_path", default="",
                        help="기본값은 체크포인트 폴더의 tfidf.pkl.")
    parser.add_argument("--tfidf_weight", type=float, default=0.10,
                        help="0이면 TF-IDF를 섞지 않습니다.")
    parser.add_argument("--batch_size", type=int, default=32)
    parser.add_argument("--device", choices=("cuda", "cpu"))
    args = parser.parse_args()

    device = torch.device(args.device or ("cuda" if torch.cuda.is_available() else "cpu"))
    names, texts = read_inputs(Path(args.label_dir))
    print(f"통화 {len(names):,}개 · 장치 {device}", flush=True)

    ckpt = Path(args.ckpt_path)
    checkpoints = sorted(ckpt.glob("*.ckpt")) if ckpt.is_dir() else [ckpt]
    if not checkpoints:
        raise SystemExit(f"체크포인트를 찾지 못했습니다: {ckpt}")

    total = None
    for i, path in enumerate(checkpoints, 1):
        print(f"  [{i}/{len(checkpoints)}] {path.name}", flush=True)
        p = probabilities(path, texts, device, args.batch_size)
        total = p if total is None else total + p
    probs = total / len(checkpoints)

    tfidf = Path(args.tfidf_path) if args.tfidf_path else \
        (ckpt if ckpt.is_dir() else ckpt.parent) / "tfidf.pkl"
    if tfidf.exists() and args.tfidf_weight > 0:
        sys.path.insert(0, str(Path(__file__).parent))
        import tfidf_model
        print(f"  TF-IDF {tfidf.name} 혼합 {args.tfidf_weight:g}", flush=True)
        probs = ((1 - args.tfidf_weight) * probs
                 + args.tfidf_weight * tfidf_model.predict(tfidf_model.load(tfidf), texts))

    output = Path(args.output)
    output.parent.mkdir(parents=True, exist_ok=True)
    with output.open("w", newline="", encoding="utf-8-sig") as handle:
        writer = csv.writer(handle)
        writer.writerow(["label file name", "symptom"])
        for name, row in zip(names, probs):
            found = [s for s, value in zip(TARGETS, row) if value >= THRESHOLD]
            writer.writerow([name, str(found)])
    print(f"저장 완료: {output}  (모델 {len(checkpoints)}개 평균, 임계값 {THRESHOLD})")


if __name__ == "__main__":
    main()

## 7. 최종 성적 (공식 Validation 3,640통화)

| 증상 | 실제 | 예측 | 정답 | Precision | Recall | **F1** |
|---|---:|---:|---:|---:|---:|---:|
| 열상 | 463 | 480 | 419 | 0.873 | 0.905 | **0.8887** |
| 복통 | 838 | 756 | 658 | 0.870 | 0.785 | **0.8256** |
| 고열 | 678 | 565 | 441 | 0.781 | 0.650 | **0.7096** |
| 호흡곤란 | 492 | 509 | 334 | 0.656 | 0.679 | **0.6673** |
| 어지러움 | 774 | 816 | 530 | 0.650 | 0.685 | **0.6667** |
| 구토 | 537 | 463 | 304 | 0.657 | 0.566 | **0.6080** |
| 전신쇠약 | 651 | 704 | 397 | 0.564 | 0.610 | **0.5860** |
| 두통 | 374 | 296 | 186 | 0.628 | 0.497 | **0.5552** |
| 오심 | 429 | 400 | 170 | 0.425 | 0.396 | **0.4101** |
| **Macro F1** | | | | | | **0.65746** |

통화당 평균 1.37개를 예측했습니다(실제 1.44개). 증상을 하나도 예측하지 않은
통화는 107개입니다.

### 계산 효율 (출제문제 권장 항목)

| 항목 | 값 |
|---|---|
| Total 파라미터 | **2,902.70M** |
| Active 파라미터 | **2,902.70M** (앙상블이므로 모든 멤버가 모든 통화를 읽습니다) |
| GPU | NVIDIA GeForce RTX 5060 Ti, VRAM 8 GiB |
| CUDA / PyTorch | 12.8 / 2.11.0+cu128 · bf16 autocast |
| Validation batch size | **32** |
| 전체 추론 시간 | **6.1분** (3,640통화) |
| 통화당 평균 | **101 ms** |

내역: base 110.63M × 7 + base(attention) 110.82M × 4 + large 336.67M × 4 +
large(attention) 336.93M × 1 + TF-IDF 1.44M.

가중치는 fp16으로 저장해 용량을 11.62 GB → **5.82 GB**로 줄였습니다. 추론
시에 fp32로 적재되므로 연산 정밀도와 무관하며, Validation 32,760개 판정 중
라벨이 달라진 것은 2건(Macro F1 차이 +0.00003)입니다.

### 선택 근거

같은 데이터에서 네 후보를 실측했습니다.

| 구성 | Macro F1 | 모델 | 추론 |
|---|---:|---:|---:|
| **16개 (제출)** | **0.65746** | 16 | 6.1분 |
| 증류 학생 6개 | 0.65683 | 6 | ~3.3분 |
| large 5개 | 0.65680 | 5 | 3.7분 |
| base 11개 | 0.65658 | 11 | 2.9분 |

차이가 최대 0.0009인데 점수 자체의 불확실성이 ±0.0059입니다(부트스트랩
4,000회). **점수로는 네 구성이 구분되지 않습니다.** 그래서 검증 완료 여부와
견고함으로 골랐습니다 — 16개 구성은 제출 전 7개 항목 점검(오프라인 적재·
오프라인 추론·CSV 규격·파일명 일치·증상 목록 범위 등)을 통과했고, 멤버가
많아 하나가 어긋나도 영향이 작으며, 검증된 11개 구성을 그대로 포함합니다.

### 규칙 준수

- 입력은 `utterances[].text`만. speaker·시간·성별·주소·중증도 미사용
- Training은 서울 데이터만. 외부 데이터 없음
- 공식 Validation은 학습·보정에 사용하지 않음
- 판정 임계값 전 증상 0.5. 증상별 임계값 없음
- 상용 AI API 미사용. 오픈소스 사전학습 체크포인트만
- 도메인 적응 MLM도 Training 전사만 사용(라벨 미사용)